# 🐍 Python Learning Series: Notebook 03
## Pydantic, Type Validation & Schema Mastery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohit-Saini-Sfdc/learn-python/blob/main/03_pydantic_mastery.ipynb)

Welcome to Notebook 03 of the **Learn Python** series!

In this interactive notebook, we explore Python's type system, why standard type hints don't enforce validation at runtime, how vanilla Python handles type casting and validation, and why **Pydantic** (`BaseModel`, `Field`, `field_validator`, `Optional`) is the industry standard for building robust, type-safe Python applications, APIs, and data pipelines.

---

In [ ]:
# Environment Setup: Ensure Pydantic v2 is installed
!pip install -q pydantic
import pydantic
print(f"Pydantic Version: {pydantic.__version__}")

## 📌 Module 1: How Vanilla Python Handles Types & Type Casting

Python is a **dynamically typed** language. While Python 3.5+ introduced **Type Hints** (`x: int`), these annotations are purely documentation and metadata for IDEs and static type checkers like Mypy. **Python's runtime ignores type hints completely.**

Let's see what happens without Pydantic when passing incorrect types or attempting manual casting.

In [ ]:
def process_user_id(user_id: int) -> int:
    """Standard function with type hints."""
    return user_id * 2

# Passing a string instead of an int does NOT throw a TypeError at runtime!
result_str = process_user_id("100")
print(f"Result with string '100': {result_str!r} (Type: {type(result_str).__name__})")

result_int = process_user_id(100)
print(f"Result with int 100: {result_int!r} (Type: {type(result_int).__name__})")

In [ ]:
# Attempting manual type casting and validation in Vanilla Python
def create_user_vanilla(user_id, age, is_active=True):
    # Manual coercion attempt
    try:
        user_id = int(user_id)
    except (ValueError, TypeError) as e:
        raise TypeError(f"user_id must be convertible to int, got {user_id!r}") from e
    
    try:
        age = int(age)
        if age < 0 or age > 120:
            raise ValueError(f"age must be between 0 and 120, got {age}")
    except (ValueError, TypeError) as e:
        raise ValueError(f"Invalid age: {e}") from e

    return {"user_id": user_id, "age": age, "is_active": bool(is_active)}

# Test valid & invalid cases
print("Valid user:", create_user_vanilla("42", "25"))

try:
    create_user_vanilla("not_an_int", "25")
except (TypeError, ValueError) as err:
    print("Caught expected error:", err)

try:
    create_user_vanilla("42", "150")
except (TypeError, ValueError) as err:
    print("Caught expected error:", err)

## 📌 Module 2: Why Pydantic is Necessary

As shown in Module 1, vanilla Python requires writing endless `try-except` blocks, `isinstance()` checks, and manual bounds assertions just to validate basic data. For complex objects, API payloads, or databases, manual validation quickly becomes unmaintainable.

### Why Pydantic?
1. **Automatic Type Parsing & Coercion:** String `"42"` automatically converts to `int(42)`.
2. **Runtime Enforced Validation:** Invalid data immediately raises a structured `ValidationError`.
3. **Zero Boilerplate:** Constraints (`ge=0`, `le=120`, regex, ISO dates) declared inline.
4. **JSON Serialization:** Convert models to/from dicts and JSON with `.model_dump()` and `.model_dump_json()`.
5. **IDE Autocomplete & Tooling:** High editor support with full type awareness.

| Feature | Vanilla Dict / Class | `@dataclass` | Pydantic `BaseModel` |
| :--- | :---: | :---: | :---: |
| Type Hints | ❌ Metadata only | ❌ Metadata only | ✅ **Runtime Enforced** |
| Auto Data Coercion | ❌ Manual | ❌ Manual | ✅ **Automatic** |
| Range & Format Constraints | ❌ Manual code | ❌ Manual `__post_init__` | ✅ **Built-in (`Field`)** |
| Serialization (`dict`/`json`) | ❌ Custom methods | ❌ `asdict()` | ✅ **Native (`.model_dump()`)** |

## 📌 Module 3: Deep Dive into `BaseModel` & `Optional` 

### `BaseModel`
`BaseModel` is the fundamental building block of Pydantic. Any class inheriting from `BaseModel` automatically gains validation, parsing, and serialization capabilities.

### `Optional[T]` vs `Optional[T] = None`
* `Optional[T]` (or `T | None`) means the value can be of type `T` **OR** `None`.
* `= None` sets the default value so the field is not required during initialization.

In [ ]:
from typing import Optional
from pydantic import BaseModel, ValidationError

class UserProfile(BaseModel):
    user_id: int                                # Required int (auto-coerced from string if valid)
    username: str                               # Required str
    email: Optional[str] = None                 # Optional field, defaults to None if omitted
    age: Optional[int] = None                   # Optional field, defaults to None

# 1. Instantiation with type coercion ("101" -> 101)
user1 = UserProfile(user_id="101", username="rohit_dev", age="28")
print("User 1 Object:", user1)
print(f"user_id type: {type(user1.user_id).__name__}") # int

# 2. Omitted optional fields default to None
user2 = UserProfile(user_id=102, username="sarah_data")
print("User 2 Object:", user2)
print(f"email: {user2.email!r}, age: {user2.age!r}")

# 3. Serialization to dict and JSON
print("Dict export:", user2.model_dump())
print("JSON export:", user2.model_dump_json())

# 4. Handling validation failures
try:
    UserProfile(user_id="invalid_id", username="test")
except ValidationError as e:
    print("\nValidationError Caught:")
    print(e)

## 📌 Module 4: Deep Dive into `Field`

`Field` is used to customize attribute behavior, apply validation rules (bounds, string length, regex patterns), set defaults/default factories, and attach documentation metadata.

### Common `Field` Parameters:
* **Numeric Constraints:** `ge` (>=), `le` (<=), `gt` (>), `lt` (<), `multiple_of`
* **String Constraints:** `min_length`, `max_length`, `pattern` (Regex match)
* **Defaults:** `default`, `default_factory` (e.g. `list`, `dict`, `datetime.now`)
* **Metadata:** `description`, `examples`, `alias`

In [ ]:
from datetime import datetime, timezone
from pydantic import BaseModel, Field, ValidationError

class CityMetrics(BaseModel):
    city_name: str = Field(min_length=2, max_length=50, description="Name of the city")
    country_code: str = Field(pattern=r"^[A-Z]{2}$", description="ISO 2-letter country code")
    latitude: float = Field(ge=-90.0, le=90.0, description="Latitude bound [-90, 90]")
    longitude: float = Field(ge=-180.0, le=180.0, description="Longitude bound [-180, 180]")
    temperature_c: float = Field(gt=-100.0, lt=70.0, description="Temperature in °C")
    humidity_pct: int = Field(ge=0, le=100, description="Humidity percentage [0, 100]")
    tags: list[str] = Field(default_factory=list, description="Dynamic list of tags")
    timestamp: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))

# Valid Record
valid_metric = CityMetrics(
    city_name="London",
    country_code="GB",
    latitude=51.5074,
    longitude=-0.1278,
    temperature_c=18.5,
    humidity_pct=72
)
print("Valid Metric:", valid_metric.model_dump_json(indent=2))

# Invalid Record (triggering multiple field validation errors)
try:
    CityMetrics(
        city_name="L",           # Too short (< 2)
        country_code="ENG",      # Regex pattern fail (not 2 uppercase chars)
        latitude=150.0,          # Out of bound (> 90.0)
        longitude=-0.1278,
        temperature_c=18.5,
        humidity_pct=150         # Out of bound (> 100)
    )
except ValidationError as e:
    print("\nField Validation Errors:")
    for err in e.errors():
        print(f" - Field '{err['loc'][0]}': {err['msg']}")

## 📌 Module 5: Deep Dive into `field_validator` & `model_validator` 

When standard constraints in `Field` are not enough, Pydantic provides custom validators:

1. **`@field_validator`**: Validates or transforms specific field(s).
   * `mode="before"`: Runs **before** Pydantic's default type parsing (receives raw input).
   * `mode="after"` (default): Runs **after** Pydantic has validated/casted the field type.
2. **`@model_validator(mode="after")`**: Validates relationships across **multiple fields** after the whole model has been constructed.

In [ ]:
from typing import Optional
from datetime import datetime, timezone
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError

class WeatherStationReading(BaseModel):
    station_id: str
    timestamp_utc: str = Field(description="ISO 8601 UTC timestamp")
    temp_celsius: float
    feels_like_celsius: float

    # 1. field_validator mode="before": Pre-process raw timestamp input
    @field_validator("timestamp_utc", mode="before")
    @classmethod
    def ensure_timestamp(cls, value: Optional[str]) -> str:
        """If timestamp is missing or empty, generate current UTC timestamp."""
        if not value or not str(value).strip():
            return datetime.now(timezone.utc).isoformat()
        return str(value).strip()

    # 2. field_validator mode="after": Sanitize string station_id
    @field_validator("station_id", mode="after")
    @classmethod
    def uppercase_station_id(cls, value: str) -> str:
        if not value.replace("_", "").isalnum():
            raise ValueError("station_id must be alphanumeric (underscores allowed)")
        return value.upper()

    # 3. model_validator mode="after": Cross-field check between temp and feels_like
    @model_validator(mode="after")
    def validate_temperature_delta(self) -> 'WeatherStationReading':
        """Ensure feels_like temperature is physically reasonable relative to temp."""
        delta = abs(self.temp_celsius - self.feels_like_celsius)
        if delta > 30.0:
            raise ValueError(
                f"Apparent temperature ({self.feels_like_celsius}°C) differs suspiciously "
                f"from actual temperature ({self.temp_celsius}°C) by > 30°C"
            )
        return self

# Test automatic timestamp generation & station ID upper-casing
reading1 = WeatherStationReading(
    station_id="st_tokyo_01",
    timestamp_utc="",  # Empty string triggers mode="before" validator
    temp_celsius=25.0,
    feels_like_celsius=27.2
)
print("Processed Reading 1:", reading1.model_dump())

# Test model validator failure (feels_like temperature way too far off)
try:
    WeatherStationReading(
        station_id="ST02",
        timestamp_utc="2026-09-09T06:00:00Z",
        temp_celsius=20.0,
        feels_like_celsius=60.0 # 40 degrees higher!
    )
except ValidationError as e:
    print("\nModel Validator Error:")
    print(e)

## 📌 Module 6: Real-World Comparison Benchmark

Let's compare processing an incoming batch of raw JSON records (like from a REST API or Kaggle dataset) using **Vanilla Python Dicts** versus **Pydantic `BaseModel`**.

In [ ]:
import json
from pydantic import BaseModel, Field, ValidationError

# Simulated messy API payload batch
raw_json_batch = """
[
  {"city": "Tokyo", "country": "Japan", "latitude": "35.6762", "longitude": 139.6503, "temp": 24.5, "humidity": 65},
  {"city": "Paris", "country": "France", "latitude": 48.8566, "longitude": "2.3522", "temp": "18.2", "humidity": "70"},
  {"city": "InvalidCity", "country": "X", "latitude": 999.0, "longitude": 10.0, "temp": 20.0, "humidity": 50},
  {"city": "Berlin", "country": "Germany", "latitude": 52.5200, "longitude": 13.4050, "temp": 19.0, "humidity": -10}
]
"""

records = json.loads(raw_json_batch)

# Approach A: Vanilla Python Manual Parsing & Validation
print("--- Approach A: Vanilla Python Manual Parsing ---")
valid_vanilla = []
errors_vanilla = []

for idx, item in enumerate(records):
    try:
        city = str(item.get("city"))
        country = str(item.get("country"))
        if len(country) < 2:
            raise ValueError(f"Invalid country code: {country}")
        lat = float(item["latitude"])
        if not (-90 <= lat <= 90):
            raise ValueError(f"Latitude out of bounds: {lat}")
        lon = float(item["longitude"])
        if not (-180 <= lon <= 180):
            raise ValueError(f"Longitude out of bounds: {lon}")
        temp = float(item["temp"])
        humidity = int(item["humidity"])
        if not (0 <= humidity <= 100):
            raise ValueError(f"Humidity out of bounds: {humidity}")
        
        valid_vanilla.append({"city": city, "lat": lat, "lon": lon, "temp": temp, "humidity": humidity})
    except Exception as err:
        errors_vanilla.append(f"Record {idx}: {err}")

print(f"Successfully Parsed: {len(valid_vanilla)} records")
print(f"Errors Caught ({len(errors_vanilla)}):")
for err in errors_vanilla:
    print(f" - {err}")


# Approach B: Pydantic Schema Validation
print("\n--- Approach B: Pydantic Schema Validation ---")

class CityWeatherPayload(BaseModel):
    city: str
    country: str = Field(min_length=2)
    latitude: float = Field(ge=-90.0, le=90.0)
    longitude: float = Field(ge=-180.0, le=180.0)
    temp: float
    humidity: int = Field(ge=0, le=100)

valid_pydantic = []
errors_pydantic = []

for idx, item in enumerate(records):
    try:
        model = CityWeatherPayload.model_validate(item)
        valid_pydantic.append(model)
    except ValidationError as e:
        errors_pydantic.append((idx, item.get("city"), e.error_count()))

print(f"Successfully Parsed: {len(valid_pydantic)} records")
print(f"Errors Caught ({len(errors_pydantic)}):")
for idx, city, count in errors_pydantic:
    print(f" - Record {idx} ({city}): {count} validation error(s)")

## 💡 Summary & Key Takeaways

1. **Type Hints are Hints:** Standard Python `x: int` does not convert or validate data at runtime.
2. **Type Coercion:** Pydantic safely parses inputs (e.g., `"100"` -> `100`, `"2026-09-09"` -> `datetime`) automatically.
3. **`BaseModel`**: The core foundation for defining typed, validated schemas with built-in `.model_dump()` and `.model_validate()`.
4. **`Optional[T] = None`**: Marks fields as optional in type and sets default fallback to `None` if omitted.
5. **`Field`**: Inline declarative constraints (`ge`, `le`, `min_length`, regex) and documentation metadata.
6. **`field_validator` & `model_validator`**: Custom procedural logic for data cleaning (`mode="before"`), complex assertions (`mode="after"`), and multi-field validation.